In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [5]:
# ================================================================
# DIC for IID / old BYM
# Using ONLY comp1 ∪ comp2
# ================================================================

import numpy as np
from pathlib import Path
from tqdm import tqdm
import pyreadr

from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import geopandas as gpd

# ================================================================
# CONFIG
# ================================================================

BASE_DIR = Path(r"D:\77\Research\temp\snow")
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# LOAD DATA (drop no_nbs)
# ================================================================

snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

y_full = snow.iloc[:, 2:].to_numpy()
coords_full = snow.iloc[:, :2].to_numpy()

y = np.delete(y_full, no_nbs, axis=0)      # (S, T)
coords = np.delete(coords_full, no_nbs, axis=0)

S, T = y.shape
print(f"[INFO] S={S}, T={T}")

# ================================================================
# BUILD GRAPH & FIND comp1, comp2
# ================================================================

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
D = squareform(pdist(xy))
W = (D <= 0.22).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

comp1_idx = np.where(labels == order[0])[0]
comp2_idx = np.where(labels == order[1])[0]

use_idx = np.sort(np.concatenate([comp1_idx, comp2_idx]))
print(f"[INFO] using |S*| = {len(use_idx)} points (comp1 ∪ comp2)")

# restrict data
y_use = y[use_idx, :]

# ================================================================
# TIME & COVARIATES
# ================================================================

t_raw = np.arange(1, T + 1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std(ddof=0)

cov_iid = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw / period),
    np.sin(2*np.pi*t_raw / period),
    t_trend
])   # (T, 4)

cov_bym = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw / period), np.cos(2*np.pi*t_raw / period),
    np.sin(2*np.pi*t_raw / period), np.sin(2*np.pi*t_raw / period),
    t_trend, t_trend
])   # (T, 8)

# ================================================================
# LOAD POSTERIOR SAMPLES
# ================================================================

iid01 = np.load(BASE_DIR / "ind01_noIso.npz")["all_theta"]
iid10 = np.load(BASE_DIR / "ind10_noIso.npz")["all_theta"]

bym01 = np.load(BASE_DIR / "bym01_noIso_final.npz")["all_theta"]
bym10 = np.load(BASE_DIR / "bym10_noIso_final.npz")["all_theta"]

M = iid01.shape[1]

# ================================================================
# HELPERS
# ================================================================

def slice_theta_block(theta, idx, S):
    """
    theta: (K*S, M)
    idx: indices in [0, S)
    return: (K*|idx|, M)
    """
    K = theta.shape[0] // S
    return np.vstack([theta[k*S + idx, :] for k in range(K)])

# ================================================================
# LOG-LIKELIHOOD (old IID / BYM)
# ================================================================

def loglik_old(y_sub, th01, th10, cov01, cov10):
    S_sub, T = y_sub.shape
    K = cov01.shape[1]
    ll = 0.0

    for t in range(1, T):
        eta01 = np.zeros(S_sub)
        eta10 = np.zeros(S_sub)

        for k in range(K):
            eta01 += cov01[t, k] * th01[k*S_sub:(k+1)*S_sub]
            eta10 += cov10[t, k] * th10[k*S_sub:(k+1)*S_sub]

        p01 = 1 / (1 + np.exp(-eta01))
        p10 = 1 / (1 + np.exp(-eta10))
        prob = np.where(y_sub[:, t-1] == 0, p01, 1 - p10)

        ll += np.sum(
            y_sub[:, t] * np.log(prob + 1e-12)
            + (1 - y_sub[:, t]) * np.log(1 - prob + 1e-12)
        )
    return ll

# ================================================================
# DIC COMPUTATION
# ================================================================

def compute_dic(label, theta01, theta10, cov01, cov10):
    # slice parameters to comp1 ∪ comp2
    th01_sub = slice_theta_block(theta01, use_idx, S)
    th10_sub = slice_theta_block(theta10, use_idx, S)

    ll = np.zeros(M)
    for m in tqdm(range(M), desc=f"DIC ({label})"):
        ll[m] = loglik_old(
            y_use,
            th01_sub[:, m],
            th10_sub[:, m],
            cov01, cov10
        )

    ll_bar = ll.mean()
    ll_hat = loglik_old(
        y_use,
        th01_sub.mean(axis=1),
        th10_sub.mean(axis=1),
        cov01, cov10
    )

    D_bar = -2 * ll_bar
    D_hat = -2 * ll_hat
    p_D = D_bar - D_hat

    return {
        "DIC": D_bar + p_D,
        "p_D": p_D,
        "loglik_mean": ll_bar,
        "loglik_at_mean": ll_hat
    }

# ================================================================
# RUN
# ================================================================

out_iid = compute_dic("IID (comp1 ∪ comp2)", iid01, iid10, cov_iid, cov_iid)
out_bym = compute_dic("old BYM (comp1 ∪ comp2)", bym01, bym10, cov_bym, cov_bym)

# ================================================================
# PRINT
# ================================================================

print("\n===== DIC (comp1 ∪ comp2) =====\n")

for name, out in [
    ("IID", out_iid),
    ("old BYM", out_bym)
]:
    print(name)
    for k, v in out.items():
        print(f"  {k}: {v:.3f}")
    print()


[INFO] S=1601, T=2704
[INFO] using |S*| = 1557 points (comp1 ∪ comp2)


DIC (old BYM (comp1 ∪ comp2)): 100%|██████████| 1000/1000 [07:38<00:00,  2.18it/s]



===== DIC (comp1 ∪ comp2) =====

IID
  DIC: 1273113.243
  p_D: 5665.560
  loglik_mean: -633723.842
  loglik_at_mean: -630891.062

old BYM
  DIC: 1283019.828
  p_D: 1336.733
  loglik_mean: -640841.547
  loglik_at_mean: -640173.181



In [10]:
# ================================================================
# DIC for NEW MODEL (p01 + p10)
# conditional likelihood, merged component
# ================================================================

import numpy as np
import pyreadr
import pickle
from tqdm import tqdm

# ================================================================
# LOAD DATA (same as MCMC)
# ================================================================

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0]

all_y = snow.drop(index=no_nbs).reset_index(drop=True)
y_full = all_y.iloc[:, 2:].to_numpy()

S_full, T = y_full.shape
period = 52

# ================================================================
# LOAD p01 / p10 MCMC RESULTS
# ================================================================

with open(BASE_DIR / "mcmc_merged_two_largest_components_p01.pkl", "rb") as f:
    res01 = pickle.load(f)["p01"]

with open(BASE_DIR / "mcmc_merged_two_largest_components_p10.pkl", "rb") as f:
    res10 = pickle.load(f)["p10"]

theta01 = res01["theta"]     # (K, Wbin01, S, M)
theta10 = res10["theta"]     # (K, Wbin10, S, M)
keep = res01["keep_idx"]     # merged component indices

y = y_full[keep, :]          # merged component data
S = y.shape[0]

K, Wbin01, _, M = theta01.shape
_, Wbin10, _, _ = theta10.shape

# ================================================================
# TIME & COVARIATES (exactly as MCMC)
# ================================================================

t_raw = np.arange(1, T + 1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std(ddof=0)

def build_cov(t_idx):
    return np.array([
        1.0, 1.0,
        np.cos(2*np.pi*t_idx/period),
        np.cos(2*np.pi*t_idx/period),
        np.sin(2*np.pi*t_idx/period),
        np.sin(2*np.pi*t_idx/period),
        t_trend[t_idx-1],
        t_trend[t_idx-1]
    ])

# ================================================================
# LOG-LIKELIHOOD (NEW MODEL)
# ================================================================

def loglik_new(theta01_m, theta10_m):
    ll = 0.0

    for t in range(1, T):
        cov = build_cov(t)

        # p01
        w01 = ((t) // 13) % Wbin01
        eta01 = np.zeros(S)
        for k in range(K):
            eta01 += cov[k] * theta01_m[k, w01, :]

        p01 = 1 / (1 + np.exp(-eta01))

        # p10
        w10 = ((t) // 26) % Wbin10
        eta10 = np.zeros(S)
        for k in range(K):
            eta10 += cov[k] * theta10_m[k, w10, :]

        p10 = 1 / (1 + np.exp(-eta10))

        # conditional likelihood
        idx0 = (y[:, t-1] == 0)
        idx1 = (y[:, t-1] == 1)

        ll += np.sum(
            y[idx0, t] * np.log(p01[idx0] + 1e-12)
            + (1 - y[idx0, t]) * np.log(1 - p01[idx0] + 1e-12)
        )

        ll += np.sum(
            (1 - y[idx1, t]) * np.log(p10[idx1] + 1e-12)
            + y[idx1, t] * np.log(1 - p10[idx1] + 1e-12)
        )

    return ll

# ================================================================
# DIC
# ================================================================

ll = np.zeros(M)

for m in tqdm(range(M), desc="DIC (new model)"):
    ll[m] = loglik_new(
        theta01[:, :, :, m],
        theta10[:, :, :, m]
    )

ll_bar = ll.mean()

theta01_bar = theta01.mean(axis=3)
theta10_bar = theta10.mean(axis=3)

ll_hat = loglik_new(theta01_bar, theta10_bar)

D_bar = -2 * ll_bar
D_hat = -2 * ll_hat
p_D = D_bar - D_hat
DIC = D_bar + p_D

# ================================================================
# PRINT
# ================================================================

print("\n===== NEW MODEL DIC =====")
print(f"loglik_mean      = {ll_bar:.3f}")
print(f"loglik_at_mean   = {ll_hat:.3f}")
print(f"p_D              = {p_D:.3f}")
print(f"DIC              = {DIC:.3f}")


DIC (new model): 100%|██████████| 1000/1000 [11:17<00:00,  1.48it/s]



===== NEW MODEL DIC =====
loglik_mean      = -606498.724
loglik_at_mean   = -605383.961
p_D              = 2229.526
DIC              = 1215226.973
